# pyplotrs — core plots verify pass

Each core plot type is rendered **twice**: first with **pyplotrs** (imported as
`pp`), then the same data with **matplotlib** (`plt`) directly below it as a
reference for visual checking. Use **Run All**, then compare each pyplotrs plot
against its matplotlib version.

pyplotrs cells end with a bare `fig`, which displays via `Figure._repr_png_`
(a bare figure shows inline). matplotlib cells use the `%matplotlib inline`
backend and end with `plt.show()`. Both plot **identical data** (stdlib
`random`, seeded — no numpy), and figure sizes match: pyplotrs `figsize` is in
**points**, so the matplotlib references convert with `pt2in` (points → inches;
72 pt = 1 in). The matplotlib cells never touch the RNG (they reuse variables
from the pyplotrs cell above), so the two libraries always see the same numbers.

**Core set:** line · bar · box · pie · scatter · errorbar · violin · polar ·
axhspan/axvspan.  **Also kept on `dev`:** histogram · fill_between · imshow.

For each plot, judge pyplotrs against its philosophy — sensible defaults, clean
spacing, colourblind-safe palette, crisp typography, publication-ready with zero
tuning — and against the matplotlib reference for correctness.

In [ ]:
import inspect
import math, random
import pyplotrs as pp
import matplotlib.pyplot as plt
%matplotlib inline

# Render matplotlib at pyplotrs' density (150 dpi) so the two show at the
# same on-screen size for side-by-side comparison.
plt.rcParams["figure.dpi"] = 150


def pt2in(w, h):
    """pyplotrs figsize is in points; matplotlib wants inches (72 pt = 1 in)."""
    return (w / 72, h / 72)


# Read pyplotrs' default figure size rather than hardcoding it: the two halves
# of every comparison below have to be the same size, and a default that drifts
# would quietly turn this notebook into a size comparison.
DEFAULT_FIGSIZE = inspect.signature(pp.subplots).parameters["figsize"].default


random.seed(0)
xs = [i * 0.25 for i in range(40)]
dists = [[random.gauss(mu, sd) for _ in range(80)]
         for mu, sd in [(0.0, 1.0), (1.8, 1.3), (-1.2, 0.8), (2.5, 1.6)]]
labels4 = ["alpha", "beta", "gamma", "delta"]
pos4 = [1, 2, 3, 4]
print("pyplotrs ready - body font:", pp.resolved_font_name())

## 1 · Line
Multiple series, dashed style, legend, axis labels.

In [ ]:
fig, ax = pp.subplots()
ax.line(xs, [math.sin(x) for x in xs], label="sin")
ax.line(xs, [math.sin(x) * math.exp(-0.15 * x) for x in xs],
        label="damped", linestyle="dashed")
ax.set(title="Line", xlabel="t", ylabel="amplitude")
ax.legend()
fig

In [ ]:
# matplotlib reference
fig, ax = plt.subplots(figsize=pt2in(*DEFAULT_FIGSIZE))
ax.plot(xs, [math.sin(x) for x in xs], label="sin")
ax.plot(xs, [math.sin(x) * math.exp(-0.15 * x) for x in xs],
        label="damped", linestyle="dashed")
ax.set(title="Line (matplotlib)", xlabel="t", ylabel="amplitude")
ax.legend()
plt.show()

## 2 · Bar
Categorical bars with named ticks.

In [ ]:
fig, ax = pp.subplots()
ax.bar(pos4, [5.0, 3.0, 7.5, 4.2])
ax.set(title="Bar", ylabel="value", xticks=pos4, xticklabels=labels4)
fig

In [ ]:
# matplotlib reference
fig, ax = plt.subplots(figsize=pt2in(*DEFAULT_FIGSIZE))
ax.bar(pos4, [5.0, 3.0, 7.5, 4.2])
ax.set(title="Bar (matplotlib)", ylabel="value")
ax.set_xticks(pos4, labels4)
plt.show()

## 3 · Box
Box-and-whisker over four distributions.

In [ ]:
fig, ax = pp.subplots()
ax.boxplot(dists, positions=pos4)
ax.set(title="Box", ylabel="value", xticks=pos4, xticklabels=labels4)
fig

In [ ]:
# matplotlib reference
fig, ax = plt.subplots(figsize=pt2in(*DEFAULT_FIGSIZE))
ax.boxplot(dists, positions=pos4)
ax.set(title="Box (matplotlib)", ylabel="value")
ax.set_xticks(pos4, labels4)
plt.show()

## 4 · Pie
Proportions with labels; equal aspect, frame off.

In [ ]:
fig, ax = pp.subplots()
ax.pie([35, 25, 22, 18], labels=labels4)
ax.set(title="Pie")
fig

In [ ]:
# matplotlib reference (startangle=90 to match pyplotrs' default)
fig, ax = plt.subplots(figsize=pt2in(*DEFAULT_FIGSIZE))
ax.pie([35, 25, 22, 18], labels=labels4, startangle=90)
ax.set(title="Pie (matplotlib)")
plt.show()

## 5 · Scatter (colour-mapped)
Per-point colour by a third value (`c=`), with a colorbar.

In [ ]:
random.seed(1)
n = 200
sx = [random.gauss(0.0, 1.0) for _ in range(n)]
sy = [random.gauss(0.0, 1.0) for _ in range(n)]
cv = [math.hypot(x, y) for x, y in zip(sx, sy)]
fig, ax = pp.subplots()
sc = ax.scatter(sx, sy, c=cv, size=24)
fig.colorbar(sc, label="radius")
ax.set(title="Scatter (colour-mapped)", xlabel="x", ylabel="y")
fig

In [ ]:
# matplotlib reference (reuses sx, sy, cv; note size= -> s=)
fig, ax = plt.subplots(figsize=pt2in(*DEFAULT_FIGSIZE))
sc = ax.scatter(sx, sy, c=cv, s=24)
fig.colorbar(sc, ax=ax, label="radius")
ax.set(title="Scatter (matplotlib)", xlabel="x", ylabel="y")
plt.show()

## 6 · Errorbar
Symmetric x and y error bars with caps.

In [ ]:
ex = list(range(1, 8))
ey = [1.0, 2.1, 1.7, 3.2, 2.8, 3.9, 3.5]
fig, ax = pp.subplots()
ax.errorbar(ex, ey, yerr=[0.3] * len(ex), xerr=[0.15] * len(ex))
ax.set(title="Errorbar", xlabel="x", ylabel="y")
fig

In [ ]:
# matplotlib reference (marker/capsize match pyplotrs' defaults)
fig, ax = plt.subplots(figsize=pt2in(*DEFAULT_FIGSIZE))
ax.errorbar(ex, ey, yerr=[0.3] * len(ex), xerr=[0.15] * len(ex),
            marker="o", capsize=3)
ax.set(title="Errorbar (matplotlib)", xlabel="x", ylabel="y")
plt.show()

## 7 · Violin
KDE violins over the same four distributions.

In [ ]:
fig, ax = pp.subplots()
ax.violinplot(dists, positions=pos4)
ax.set(title="Violin", ylabel="value", xticks=pos4, xticklabels=labels4)
fig

In [ ]:
# matplotlib reference
fig, ax = plt.subplots(figsize=pt2in(*DEFAULT_FIGSIZE))
ax.violinplot(dists, positions=pos4)
ax.set(title="Violin (matplotlib)", ylabel="value")
ax.set_xticks(pos4, labels4)
plt.show()

## 8 · Polar
Line on a polar projection (radians, CCW from East), with a legend.

In [ ]:
theta = [i * math.pi / 180 for i in range(361)]
fig, ax = pp.subplots(projection="polar", figsize=(360, 360))
ax.plot(theta, [abs(math.cos(2 * t)) for t in theta], label="rose")
ax.plot(theta, [t / (2 * math.pi) for t in theta], label="spiral", linestyle="dashed")
ax.set(title="Polar")
ax.legend()
fig

In [ ]:
# matplotlib reference (reuses theta)
fig, ax = plt.subplots(subplot_kw={"projection": "polar"}, figsize=pt2in(360, 360))
ax.plot(theta, [abs(math.cos(2 * t)) for t in theta], label="rose")
ax.plot(theta, [t / (2 * math.pi) for t in theta], label="spiral", linestyle="dashed")
ax.set(title="Polar (matplotlib)")
ax.legend()
plt.show()

## 9 · axhspan / axvspan
Shaded bands + reference lines over a line.

In [ ]:
fig, ax = pp.subplots()
ax.line(xs, [math.sin(x) for x in xs])
ax.axhspan(0.5, 1.0, color="C1", alpha=0.15)
ax.axvspan(2.0, 4.0, color="C2", alpha=0.15)
ax.axhline(0.0)
ax.axvline(5.0, linestyle="dashed")
ax.set(title="axhspan / axvspan + reference lines", xlabel="t", ylabel="sin")
fig

In [ ]:
# matplotlib reference
fig, ax = plt.subplots(figsize=pt2in(*DEFAULT_FIGSIZE))
ax.plot(xs, [math.sin(x) for x in xs])
ax.axhspan(0.5, 1.0, color="C1", alpha=0.15)
ax.axvspan(2.0, 4.0, color="C2", alpha=0.15)
ax.axhline(0.0)
ax.axvline(5.0, linestyle="dashed")
ax.set(title="axhspan / axvspan (matplotlib)", xlabel="t", ylabel="sin")
plt.show()

## Also kept · Histogram

In [ ]:
hsample = [random.gauss(0.0, 1.0) for _ in range(1000)]
fig, ax = pp.subplots()
ax.hist(hsample, bins=24)
ax.set(title="Histogram", xlabel="value", ylabel="count")
fig

In [ ]:
# matplotlib reference (reuses hsample)
fig, ax = plt.subplots(figsize=pt2in(*DEFAULT_FIGSIZE))
ax.hist(hsample, bins=24)
ax.set(title="Histogram (matplotlib)", xlabel="value", ylabel="count")
plt.show()

## Also kept · fill_between

In [ ]:
base = [math.sin(x) for x in xs]
lo = [v - 0.3 for v in base]
hi = [v + 0.3 for v in base]
fig, ax = pp.subplots()
ax.fill_between(xs, lo, hi, label="band")
ax.line(xs, base, label="mean")
ax.set(title="fill_between", xlabel="t", ylabel="y")
ax.legend()
fig

In [ ]:
# matplotlib reference (reuses base, lo, hi)
fig, ax = plt.subplots(figsize=pt2in(*DEFAULT_FIGSIZE))
ax.fill_between(xs, lo, hi, label="band")
ax.plot(xs, base, label="mean")
ax.set(title="fill_between (matplotlib)", xlabel="t", ylabel="y")
ax.legend()
plt.show()

## Also kept · imshow (+ colorbar)

In [ ]:
grid = [[math.sin(0.3 * i) * math.cos(0.3 * j) for j in range(30)] for i in range(20)]
fig, ax = pp.subplots()
im = ax.imshow(grid, cmap="viridis")
fig.colorbar(im, label="value")
ax.set(title="imshow")
fig

In [ ]:
# matplotlib reference (reuses grid)
fig, ax = plt.subplots(figsize=pt2in(*DEFAULT_FIGSIZE))
im = ax.imshow(grid, cmap="viridis")
fig.colorbar(im, ax=ax, label="value")
ax.set(title="imshow (matplotlib)")
plt.show()